# Preprocessing (1971–1986)

In [ ]:
import pandas as pd
import openpyxl
import re
import os
import swifter
import spacy
import string
import nltk

from ocrfixr import spellcheck
from nltk.tokenize import sent_tokenize,word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from matplotlib import pyplot
from langdetect import detect
from googletrans import Translator
from spacy.lang.en import English

# nlp_en = spacy.load('en_core_web_sm')
# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')A

In [ ]:
# !jupyter --version
# !jupyter labextension install @jupyter-widgets/jupyterlab-manager

### Fully preprocessed variant (word-level): 1971–1986

In [ ]:
data_1971_1986_df = pd.read_pickle("../../datasets/data_split_timeframe/data_1971-1986.pkl")

In [ ]:
# stopword removal and lemmatization

def remove_stopwords_and_save(text):
    # Initialize WordNet lemmatizer and stopwords
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))
    punctuation = set(string.punctuation)

    # Tokenize the text
    tokens = nltk.word_tokenize(text)

    # Remove stopwords and lemmatize the words longer than 2 characters and punctuation,
    # clean_words = [lemmatizer.lemmatize(word) if word not in punctuation else word for word in tokens if word.lower() not in stop_words and len(word) > 2]
    # clean_words = [lemmatizer.lemmatize(word) if len(word) > 2 and word.lower() not in stop_words else word for word in tokens]
    clean_words = [lemmatizer.lemmatize(word) for word in tokens if len(word) > 2 and word.lower() not in stop_words]



    # Join the cleaned words to form clean text
    clean_text = ' '.join(clean_words)

    
    
    # Identify removed words
    removed_words = [word for word in tokens if word.lower() in stop_words or len(word) <= 2 and word not in punctuation] 

    # Join the removed words to form removed text
    removed_text = ' '.join(removed_words)

    return clean_text, removed_text


# Apply the lambda function to remove stopwords from the 'contents' column
data_1971_1986_df['content_translated_processed_1'], data_1971_1986_df['removed_text'] = zip(*data_1971_1986_df['content_translated_processed'].swifter.apply(remove_stopwords_and_save))

In [ ]:

def cleaning_reg(text):
    removed_words=[]
    text= re.sub(r'\.{2,}', '.', text) # repeated full stop pattern
    text= re.sub(r'\b(\w)\1+\b','', text) # repeated words pattern
    text = re.sub(r'([a-z])\1{2,}', '', text) # repeated letter
    text = re.sub(r'([\W+])\1{1,}', r'\1', text) # non-word repitition
    text = re.sub(r'(.{2,}?)\1{1,}', r'\1', text) # phrase repitition
    # Apply regex pattern to remove unwanted numbers
    # text = re.sub(r'(?!\b(?:\d{1,2}/\d{1,2}/\d{2,4}|(?:19|20)\d{2})\b|\b\d{1,2}\.\s+(?:January|February|March|April|May|June|July|August|September|October|November|December|Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)\b)\d+\b', '', text)
    # text = re.sub(r'\b(?!\d{1,2}/\d{1,2}/\d{2,4}(?!\d)|\b(?:19|20)\d{2}(?!\d))\d+\b', '', text)
    # text = re.sub(r'(?!\d{1,2}/\d{1,2}/\d{2,4}(?!\d)|\b(?:19|20)\d{2}(?!\d)|\b\d+\.\s+(?:January|February|March|April|May|June|July|August|September|October|November|December))\d+\b', '', text)
    
    # remove continuous dots or full stops (.   .  .   .   .   .  .   .   .     .   .   .   .  .   .  )
    # text = re.sub(r'\.\s*\.\s*\.\s*', '', text)
    text = re.sub(r',\s*,+', ' ', text)
    #remove extra symbols
    text = re.sub(r'[;?]', '', text)
    #remove extra white space
    text = re.sub(r'\s+', ' ', text)
    # Remove numbers and save removed numbers
    removed_numbers = re.findall(r'\b\d+\b', text)
    text = re.sub(r'\b\d+\b', '', text)
    #Remove other punctuation marks except full stops
    text = re.sub(r'[^\w\s.]', '', text)

    # Track removed words (less than or equal to two characters )
    removed_words.extend(re.findall(r'\b\w{1,2}\b', text))
    text = re.sub(r'\b\w{1,2}\b', '', text)
    return text,removed_words, removed_numbers

In [ ]:
# Define a function to process each batch
def process_batch(batch):
    batch['content_translated_processed_deep'], batch['removed_words'], batch['removed_numbers'] = zip(*batch['content_translated_processed_1'].swifter.apply(cleaning_reg))
    return batch

# Split the DataFrame into smaller batches
batch_size = 1000  # Adjust the batch size as needed
batches = [data_1971_1986_df[i:i+batch_size] for i in range(0, len(data_1971_1986_df), batch_size)]

# Process each batch and concatenate the results
processed_batches = [process_batch(batch) for batch in batches]
result_df = pd.concat(processed_batches)

In [ ]:
result_df.to_pickle("../../datasets/data_split_timeframe/data_1971_1986_cleaned_df_regex.pkl")

In [ ]:
data_1971_1986_cleaned_df_regex= pd.read_pickle("../../datasets/data_split_timeframe/data_1971_1986_cleaned_df_regex.pkl")

#### OCR spell correction with OCRFixr (not used)

In [ ]:
data_1971_1986_cleaned_df_regex= pd.read_pickle("../../datasets/data_split_timeframe/data_1971_1986_cleaned_df_regex.pkl")

In [ ]:
# # implement ocrfixr

# def correct_and_track_misspelled(text, chunk_size):
#     sentences = sent_tokenize(text)
#     corrected_text = ""  # Initialize variable to accumulate corrected chunks
#     misspelled_words = {}  # Initialize dictionary to accumulate misspelled words
    
#     for sentence in sentences:
#         words = word_tokenize(sentence)
#         current_chunk = ""
        
#         for word in words:
#             if len(current_chunk) + len(word) + 1 <= chunk_size:
#                 current_chunk += word + " "
#             else:
#                 fix = spellcheck(current_chunk.strip(), return_fixes='T').fix()
#                 corrected_text += fix[0] + " "  # Append corrected chunk to the accumulated corrected text
#                 misspelled_words.update(fix[1])  # Update misspelled words dictionary
#                 current_chunk = word + " "
        
#         if current_chunk:
#             fix = spellcheck(current_chunk.strip(), return_fixes='T').fix()
#             corrected_text += fix[0] + " "  # Append corrected chunk to the accumulated corrected text
#             misspelled_words.update(fix[1])  # Update misspelled words dictionary
    
#     return corrected_text.strip(), misspelled_words

# chunk_size = 500 #(since it uses BERT, we use chunk here)

# # Assuming your DataFrame is named df and the column containing documents is named 'content_translated_processed'
# data_1971_1986_cleaned_df_regex[['ocr_corrected_document', 'ocr_misspelled_words']] = data_1971_1986_cleaned_df_regex['content_translated_processed_deep'].swifter.apply(lambda text: pd.Series(correct_and_track_misspelled(text, chunk_size)))

#### Lemmatisation: comparison of NLTK and spaCy

In [ ]:
# Initialize NLTK Lemmatizer
lemmatizer = WordNetLemmatizer()

# Initialize spaCy
nlp = spacy.load("en_core_web_sm")

# Define function to lemmatize using NLTK
def nltk_lemmatize(text):
    tokens = nltk.word_tokenize(text)
    lemmatized_text = ' '.join([lemmatizer.lemmatize(word) for word in tokens])
    return lemmatized_text

# # Define function to capture differences between NLTK and spaCy lemmatization
# def capture_differences(nltk_list, spacy_list):
#     differing_words = []
#     for word_nltk, word_spacy in zip(nltk_list, spacy_list):
#         if word_nltk != word_spacy:
#             differing_words.append((word_nltk, word_spacy))
#     return differing_words

def capture_differences(nltk_doc, spacy_doc):
    differing_words = []
    for word_nltk, word_spacy in zip(nltk_doc.split(), spacy_doc.split()):
        if word_nltk != word_spacy:
            differing_words.append((word_nltk, word_spacy))
    return differing_words


# Apply spaCy lemmatization to each row of the DataFrame
data_1971_1986_cleaned_df_regex['content_cleaned_spacy_lemmatized'] = data_1971_1986_cleaned_df_regex['content_translated_processed_deep'].swifter.apply(lambda x: " ".join([token.lemma_ for token in nlp(x)]))

# Apply NLTK lemmatization to each row of the DataFrame
data_1971_1986_cleaned_df_regex['content_cleaned_nltk_lemmatized'] = data_1971_1986_cleaned_df_regex['content_translated_processed_deep'].swifter.apply(nltk_lemmatize)

# to capture differences between NLTK and spaCy lemmatization
data_1971_1986_cleaned_df_regex['differing_words'] = data_1971_1986_cleaned_df_regex.swifter.apply(lambda row: capture_differences(row['content_cleaned_spacy_lemmatized'], row['content_cleaned_nltk_lemmatized']), axis=1)

In [ ]:
data_1971_1986_cleaned_df_regex.to_pickle("../../datasets/data_split_timeframe/data_1971_1986_cleaned_df_regex_lemma.pkl")

In [ ]:
data_1971_1986_cleaned_df_regex_lemma=pd.read_pickle("../../datasets/data_split_timeframe/data_1971_1986_cleaned_df_regex_lemma.pkl")

In [ ]:
# remove stopwords using spacy

# Define a lambda function to remove stop words from a document and capture the removed words
remove_stopwords_spacy = lambda text: (
    " ".join([token.text for token in nlp(text) if not token.is_stop]),  # cleaned text
    " ".join([token.text for token in nlp(text) if token.is_stop])       # removed stop words
)
data_1971_1986_cleaned_df_regex_lemma[['content_cleaned_spacy_lemmatized' + '_cleaned', 'content_cleaned_spacy_lemmatized' + '_removed']] = data_1971_1986_cleaned_df_regex_lemma['content_cleaned_spacy_lemmatized'].swifter.apply(remove_stopwords_spacy).apply(pd.Series)

In [ ]:
translator = str.maketrans('', '', string.punctuation)
data_1971_1986_cleaned_df_regex_lemma['content_cleaned_spacy_lemmatized_cleaned'] = data_1971_1986_cleaned_df_regex_lemma['content_cleaned_spacy_lemmatized_cleaned'].swifter.apply(lambda x: x.translate(translator))

In [ ]:
data_1971_1986_cleaned_df_regex_lemma.to_pickle("../../datasets/data_split_timeframe/data_1971_1986_cleaned_df_regex_lemma_stop.pkl")

In [ ]:
data_1971_1986_cleaned_df_regex_lemma=pd.read_pickle('../../datasets/data_split_timeframe/data_1971_1986_cleaned_df_regex_lemma_stop.pkl')

In [ ]:
# Dataset = data_1971_1986_cleaned_df_regex_lemma_stop.pkl

### Lightly cleaned variant (sentence-level): 1971–1986

In [ ]:
data_1971_1986_sent_df = pd.read_pickle("../../datasets/data_split_timeframe/data_1971-1986.pkl")

In [ ]:
def cleaning_reg_sent(text):
    text= re.sub(r'\.{2,}', '', text) # repeated full stop pattern
    text= re.sub(r'\b(\w)\1+\b','', text) # repeated words pattern
    text = re.sub(r'([a-z])\1{2,}', '', text) # repeated letter
    text = re.sub(r'([\W+])\1{1,}', r'\1', text) # non-word repitition
    text = re.sub(r'(.{2,}?)\1{1,}', r'\1', text) # phrase repitition
    # remove continuous dots or full stops (.   .  .   .   .   .  .   .   .     .   .   .   .  .   .  )
    # text = re.sub(r'\.\s*\.\s*\.\s*', '', text)
    text = re.sub(r',\s*,+', ' ', text)
    #remove extra symbols
    text = re.sub(r'[;?]', '', text)
    #remove extra white space
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s.\d]', '', text) # any character that is not a word character, whitespace character, full stop, or number, and remove them

    return text

In [ ]:
# Define a function to process each batch
def process_batch(batch):
    batch['content_translated_processed_regex']= batch['content_translated_processed'].swifter.apply(cleaning_reg_sent)
    return batch

# Split the DataFrame into smaller batches
batch_size = 1000
batches = [data_1971_1986_sent_df[i:i+batch_size] for i in range(0, len(data_1971_1986_sent_df), batch_size)]

# Process each batch and concatenate the results
processed_batches = [process_batch(batch) for batch in batches]
result_df = pd.concat(processed_batches)

In [ ]:

result_df.to_pickle("../../datasets/data_split_timeframe/data_1971_1986_sent_df_regex.pkl")

In [ ]:

data_1971_1986_sent_df_regex= pd.read_pickle("../../datasets/data_split_timeframe/data_1971_1986_sent_df_regex.pkl")

In [ ]:
# Load spaCy English model
nlp = spacy.load('en_core_web_sm')

# Function to clean text using spaCy
def clean_text_spacy(text):
    # Process the text using spaCy
    doc = nlp(text)
    # Keep only the tokens with a length of more than 1 character
    cleaned_text = ' '.join(token.text.lower() for token in doc if len(token.text) >= 2 or token.is_punct or token.is_digit)
    return cleaned_text

# Apply the clean_text_spacy function to a DataFrame column
data_1971_1986_sent_df_regex['content_translated_processed_regex_cleaned'] = data_1971_1986_sent_df_regex['content_translated_processed_regex'].swifter.apply(clean_text_spacy)

In [ ]:
data_1971_1986_sent_df_regex.to_pickle("../../datasets/data_split_timeframe/data_1971_1986_sent_df_regex_cleaned.pkl")

In [ ]:
data_1971_1986_sent_df_regex_cleaned=pd.read_pickle("../../datasets/data_split_timeframe/data_1971_1986_sent_df_regex_cleaned.pkl")

In [ ]:
# Dataset - data_1971_1986_sent_df_regex_cleaned.pkl